# Assignment 02 – NumPy, pandas and Your Dataset
**Domain:** Cybersecurity – Clinic Security Monitoring  
**Dataset:** PhiUSIIL Phishing URL (Website), UCI Machine Learning Repository, dataset ID 967.

The dataset contains URL and webpage-related records. UCI reports 235,795 instances and 54 features; label 1 means legitimate and label 0 means phishing. Source: https://archive.ics.uci.edu/dataset/967/phiusiil+phishing+url+dataset

## 1. Load the real dataset
This cell downloads the official dataset through the UCI repository interface and loads it into a pandas DataFrame. It should exceed the required 500 rows and five columns.

In [ ]:
# Load the official UCI PhiUSIIL phishing URL dataset.
# UCI label convention: 1 = legitimate, 0 = phishing.
!pip -q install ucimlrepo

import pandas as pd
import numpy as np
from ucimlrepo import fetch_ucirepo

dataset = fetch_ucirepo(id=967)
df = pd.concat([dataset.data.features, dataset.data.targets], axis=1)
df.columns = [str(c).strip() for c in df.columns]
print("Dataset shape:", df.shape)
print("Columns:", df.columns.tolist())
print("Label column candidates:", [c for c in df.columns if c.lower() == "label"])
display(df.head())

## 2. Dataset dimensions, column types and missing values
Answer the required inspection questions using code rather than estimates.

In [ ]:
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

print("\nColumn data types:")
display(df.dtypes.to_frame("data_type"))

print("\nMissing values per column:")
display(df.isna().sum().to_frame("missing_values"))

print("\nNumeric summary (minimum, maximum, mean, median):")
numeric_df = df.select_dtypes(include=np.number)
summary = pd.DataFrame({
    "minimum": numeric_df.min(),
    "maximum": numeric_df.max(),
    "mean": numeric_df.mean(),
    "median": numeric_df.median()
})
display(summary)

## 3. Text column: distinct values and five most common values
The raw URL column is categorical/text data. This cell identifies it safely and reports its distinct count and most frequent values.

In [ ]:
text_candidates = df.select_dtypes(include=["object", "string"]).columns.tolist()
print("Text columns:", text_candidates)

url_col = next((c for c in df.columns if c.lower() == "url"), None)
if url_col is None:
    raise ValueError("Expected URL text column was not found.")
print("Distinct URL values:", df[url_col].nunique(dropna=True))
print("Five most common URL values:")
display(df[url_col].value_counts().head(5).to_frame("count"))

## 4. Selecting and filtering
Demonstrate one column selection, two-column selection, and a domain-meaningful filter. UCI defines label 0 as phishing and label 1 as legitimate.

In [ ]:
label_col = next(c for c in df.columns if c.lower() == "label")

# One column
display(df[url_col].head())

# Two columns together
display(df[[url_col, label_col]].head())

# Domain-meaningful filter: phishing-labelled records (label 0)
phishing_rows = df[df[label_col] == 0]
print("Phishing-labelled rows:", len(phishing_rows))
display(phishing_rows[[url_col, label_col]].head())

## 5. NumPy loop sum versus vectorised sum
Choose a numeric column and compute the same sum with a Python loop and NumPy. `%timeit` records both execution times in Colab.

In [ ]:
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()
numeric_col = next(c for c in numeric_columns if c.lower() != "label")
values = df[numeric_col].dropna().to_numpy(dtype=float)

def python_loop_sum(array):
    total = 0.0
    for value in array:
        total += value
    return total

loop_total = python_loop_sum(values)
numpy_total = values.sum()
print("Loop sum:", loop_total)
print("NumPy sum:", numpy_total)
print("Results match:", np.isclose(loop_total, numpy_total))

print("\nPython loop timing:")
%timeit python_loop_sum(values)
print("\nNumPy vectorised timing:")
%timeit values.sum()

## 6. Five-sentence dataset description
This dataset contains labelled examples of legitimate and phishing URLs and related website characteristics. Each row represents a URL or webpage observation, while columns describe URL structure and other extracted attributes. The `label` column identifies legitimate records with 1 and phishing records with 0. The dataset is intended for supervised classification experiments in cybersecurity. It is a benchmark dataset and does not represent every website, user, or phishing campaign in the real world.

## Completion checklist
- [ ] At least 500 rows and five columns; dataset source linked.
- [ ] Dimensions, data types, missing counts, numeric min/max/mean/median and text value counts shown with code.
- [ ] One-column selection, two-column selection and meaningful phishing filter demonstrated.
- [ ] Loop and NumPy sums match and both `%timeit` timings are recorded.
- [ ] Five-sentence plain-language dataset description included.